# Module 1, Activity 2: Prompt Engineering and Context Management

In [ ]:
#import json
#from pprint import pprint
import boto3

#from langchain_aws import ChatBedrock, ChatBedrockConverse
from langchain_aws import ChatBedrock
from langchain.chains import LLMChain
from langchain_core.output_parsers import StrOutputParser
#from langchain_core.runnables import RunnableLambda
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [ ]:
session = boto3.session.Session()
region = session.region_name

## About this cell

This is a helper function to download data from a file in an S3 bucket.  You can also make the connection directly, if you choose.

In [ ]:
def get_data_from_s3(bucket_name, key):
    s3 = boto3.client('s3', region_name=region)
    response = s3.get_object(Bucket=bucket_name, Key=key)
    return response['Body'].read().decode('utf-8')

In [ ]:
s3_data = get_data_from_s3("bdc-aws-workshop", "constitution.txt")
print(s3_data[:300])

## Zero-Shot Prompting

We are now going to pass this text into the LLM and ask it to analyze it with varying degrees of instruction, as passed through the prompt.  

The most basic type of prompting is called "zero-shot prompting," which is when you provide the model with just the question or instruction and expect it to get the answer just based on its pre-trained knowledge.  No examples or additional context are provided.  Let's try it:

In [ ]:
system_prompt = SystemMessagePromptTemplate.from_template("You are a helpful assistant.")
human_prompt = HumanMessagePromptTemplate.from_template("{input}")
prompt = ChatPromptTemplate.from_messages([system_prompt, human_prompt])

llm = ChatBedrock(
    model_id="anthropic.claude-3-sonnet-20240229-v1:0",
    region_name=region,
    temperature=0.5,
    max_tokens=1000,
)
chain = prompt | llm | StrOutputParser()
print(chain.invoke({"input": f"Provide an analysis of {s3_data}"}))

## Few-Shot Prompting

While that did alright, we can do better if we can provide a few examples within the prompt to guide the model's behavior.  This is called "few-shot prompting."  By demonstrating the format, style, or type of answer you expect, the model can better understand and mimic that structure in its response.

In [ ]:
few_shot_prompt = f"""
You are provided with {s3_data} and given the following:
Example 1:
Q: What is the significance of the Constitution of the United States?
A: The Constitution is the supreme law of the United States...

Example 2:
Q: How does the Constitution implement checks and balances?
A: It divides power among three branches...

Now, answer the following:
Q: Provide an analysis of the Constitution of the United States.
A:
"""
print(chain.invoke({"input": few_shot_prompt}))

## Chain-of-Thought (COT) Prompts

Chain-of-thought prompting is a technique where you guide the model to break down its reasoning process into sequential steps before arriving at the final answer. Instead of generating a direct answer in one go, you instruct the model to "think aloud" by detailing intermediate steps.  This can lead to more thorough and accurate responses, especially for complex or multi-step problems.  It works particularly well with more sophisticated models.

In [ ]:
cot_prompt = f"""
You are an expert in constitutional law. You have been provided {s3_data}.
Please analyze this data by following these steps:
Step 1: Summarize the structure of the Constitution.
Step 2: Explain the checks and balances.
Step 3: Discuss its modern legal influence.
"""
print(chain.invoke({"input": cot_prompt}))

In [ ]:
print(chain.invoke({"input": "What subject did we just discuss?"}))

## Oops!

Notice that we just asked the LLM to tell us about what we have been talking about with it.  However, LLM's do not, by default, have any memory.  This is something we need to add to it by creating a chat history.

_**Note:**_

It is likely when you run the below cell you will get a deprecation error.  This is because LangChain is changing how they handle conversational memory, encouraging migration to their new platform, LangGraph.  LangGraph is a sophisticated platform used for the orchestration of multi-agent bots that use tools.  At this stage it is beyond the scope of where we are in this workshop but we will discuss it when we get to "Module 4: Agents and Tool Use."  

In [ ]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
system_prompt_mem = SystemMessagePromptTemplate.from_template(
    "You are a helpful assistant. Use the conversation history: {chat_history}"
)
human_prompt_mem = HumanMessagePromptTemplate.from_template("{input}")
prompt_mem = ChatPromptTemplate.from_messages([system_prompt_mem, human_prompt_mem])

chain_with_memory = LLMChain(
    llm=llm,
    prompt=prompt_mem,
    memory=memory,
)

In [ ]:
response1 = chain_with_memory.invoke("What is the recipe for mayonnaise?")
print(response1['text'])

In [ ]:
response2 = chain_with_memory.invoke("What recipe did I just ask you for?")
print(response2['text'])

## Best Practices for Prompt Engineering

Notice that we used statements like "you are an expert in..."  Informing the LLM how they should respond within the prompt is considered to be good prompt engineering practice.  But there are many other things you should consider adding to your prompts.  Here are some general guidelines taken from [this website](https://help.openai.com/en/articles/6654000-best-practices-for-prompt-engineering-with-the-openai-api):

- Use the latest models (noting that many popular models are updated regularly...be sure you have the most recent version)
- Put the instructions at the beginning of the prompt and use delimiters like `####` or `""""` to separate the instruction and context.
- Be specific, descriptive, and as detailed as possible about the desired context, outcome, length, format, style, etc.
- Articulate the desired output format through examples
- When possible, do not use imprecise descriptions
- Don't just say what NOT to do...say what to do instead

AWS also has a great guide on prompt engineering with the Titan models that can be found [here](https://d2eo22ngex1n9g.cloudfront.net/Documentation/User+Guides/Titan/Amazon+Titan+Text+Prompt+Engineering+Guidelines.pdf).